# Emulsion Microscopy Analysis — Demo Workflow

> **Data disclaimer:** All images in this project are **synthetically generated**.
> This notebook demonstrates a reproducible image analysis workflow, not real-world model validity.

## Build progress

| Step | Status | Description |
|------|--------|-------------|
| **Step 1** | ✅ done | Synthetic data generation |
| **Step 2** | ✅ done | Classical image analysis |
| **Step 3** | ✅ this notebook | Deep learning classification |
| Step 4 | coming next | Final visualization + README polish |

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from the notebooks/ directory
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.io import imread

from src.generate_data import main as generate_data, RAW_DIR, METADATA_CSV

%matplotlib inline

## Step 1 · Generate Synthetic Images

The cell below runs `src/generate_data.py` and produces:

- **150 PNG files** in `data/raw/`  (50 per class: `small` / `medium` / `large`)
- **`data/metadata.csv`** — image filename, droplet count, mean radius, size class, seed
- **`results/synthetic_examples.png`** — 3 × 5 preview grid

Expected run time: ~10–20 seconds on CPU.

In [ ]:
generate_data()

## Dataset Overview

`metadata.csv` gives one row per image with the labels we will use in later steps.

In [ ]:
df = pd.read_csv(METADATA_CSV)
print(f'Total images : {len(df)}')
print(f'Size classes : {df["size_class"].unique().tolist()}')
display(df.head(9))

In [ ]:
print('Images per class:')
display(df['size_class'].value_counts())

print('\nDroplet stats per class (mean radius in pixels):')
display(
    df.groupby('size_class')[['droplet_count', 'mean_radius']]
    .describe()
    .round(2)
)

## Visual Examples

Four random images from each size class.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(13, 10))

for row_i, cls in enumerate(['small', 'medium', 'large']):
    subset = df[df['size_class'] == cls].sample(4, random_state=42)
    for col_i, (_, row) in enumerate(subset.iterrows()):
        img = imread(str(RAW_DIR / row['image_filename']))
        ax  = axes[row_i, col_i]
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.set_title(
            f"{cls}\nn_drops={row['droplet_count']}  r={row['mean_radius']:.1f}px",
            fontsize=9
        )
        ax.axis('off')

plt.suptitle('Synthetic Emulsion-like Images  (Step 1)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Radius Distribution by Class

Confirms the three classes have distinct, non-overlapping mean radius ranges —
a clean separation that gives the later classifier a fair learning signal.

In [ ]:
colors = {'small': '#4C9BE8', 'medium': '#E8A44C', 'large': '#5EBD70'}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for cls, grp in df.groupby('size_class'):
    axes[0].hist(grp['mean_radius'], bins=15, alpha=0.7,
                 label=cls, color=colors[cls], edgecolor='white')
axes[0].set_xlabel('Mean droplet radius (px)')
axes[0].set_ylabel('Image count')
axes[0].set_title('Radius Distribution per Class')
axes[0].legend()

for cls, grp in df.groupby('size_class'):
    axes[1].hist(grp['droplet_count'], bins=range(1, 12), alpha=0.7,
                 label=cls, color=colors[cls], edgecolor='white')
axes[1].set_xlabel('Droplet count per image')
axes[1].set_ylabel('Image count')
axes[1].set_title('Droplet Count Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Step 1 complete.')
print('Next: Step 2 - classical image analysis (thresholding + connected components).')

---

## Step 2 · Classical Image Analysis

Pipeline: **Gaussian blur → Otsu threshold → remove small objects → connected components → region properties**

No hand-tuned threshold values — Otsu finds the cutoff automatically from the image histogram.

In [ ]:
from src.classical import detect_droplets, analyse_dataset, save_detection_overlay, save_size_distribution, RESULTS_DIR

### Single-image walkthrough

Let's trace each pipeline step on one example image so the logic is visible.

In [ ]:
from skimage import filters, morphology, measure

# pick one representative image from each class for the walkthrough
examples = {
    cls: imread(str(RAW_DIR / df[df['size_class'] == cls].iloc[2]['image_filename']))
    for cls in ['small', 'medium', 'large']
}

fig, axes = plt.subplots(3, 4, figsize=(14, 11))
step_titles = ['Original', 'After blur + Otsu', 'After cleanup', 'Detected blobs']

for row_i, (cls, img) in enumerate(examples.items()):
    blurred = filters.gaussian(img, sigma=1.5, preserve_range=True)
    thresh  = filters.threshold_otsu(blurred)
    binary  = blurred > thresh
    cleaned = morphology.remove_small_objects(binary, min_size=30)
    cleaned = morphology.remove_small_holes(cleaned, area_threshold=30)
    labeled = measure.label(cleaned)
    regions = measure.regionprops(labeled)

    stages = [img, binary, cleaned, img]
    cmaps  = ['gray', 'gray', 'gray', 'gray']

    for col_i, (stage, cmap) in enumerate(zip(stages, cmaps)):
        ax = axes[row_i, col_i]
        ax.imshow(stage, cmap=cmap, vmin=0, vmax=(255 if col_i == 0 or col_i == 3 else 1))
        ax.set_title(f'{cls} · {step_titles[col_i]}', fontsize=8)
        ax.axis('off')

        # on the last column, overlay detected circles
        if col_i == 3:
            for reg in regions:
                cy, cx = reg.centroid
                r = reg.equivalent_diameter / 2
                ax.add_patch(plt.Circle((cx, cy), r,
                                        edgecolor='lime', linewidth=1.5, fill=False))
            ax.set_title(f'{cls} · {len(regions)} blobs found', fontsize=8)

plt.suptitle('Step 2 · Pipeline walkthrough (one image per class)', fontsize=12)
plt.tight_layout()
plt.show()

### Detection overlay — saved to `results/`

Runs across all 150 images and saves `results/classical_detection_overlay.png`.

In [ ]:
save_detection_overlay()

from IPython.display import Image as IPImage
IPImage(str(RESULTS_DIR / 'classical_detection_overlay.png'))

### Feature extraction — full dataset

`analyse_dataset()` returns a DataFrame with one row per detected blob.
Columns: `image_filename`, `size_class`, `area_px`, `equiv_diameter_px`, `solidity`.

In [ ]:
results_df = analyse_dataset()

print(f'Total blobs detected: {len(results_df)}')
display(results_df.head(9))

print('\nMean equivalent diameter per class:')
display(results_df.groupby('size_class')['equiv_diameter_px'].describe().round(2))

### Size distribution — saved to `results/`

In [ ]:
save_size_distribution(results_df)

IPImage(str(RESULTS_DIR / 'droplet_size_distribution.png'))

In [ ]:
print('Step 2 complete.')
print('Next: Step 3 — deep learning classification (simple CNN).')

---

## Step 3 · Deep Learning Classification

We train a **simple 2-layer CNN** (no pretrained weights) to classify 64×64 droplet patches into `small / medium / large`.

Pipeline:
1. Crop patches around every droplet detected by the classical pipeline
2. Split into train / val / test (stratified by class)
3. Train SimpleCNN for 20 epochs on CPU
4. Evaluate: training curve, confusion matrix, prediction examples

> This is a **demonstrative baseline**, not a production model.

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.dataset import (
    build_patch_dataset, make_splits,
    CLASS_TO_IDX, IDX_TO_CLASS, CLASS_ORDER, PATCHES_DIR
)
from src.model import SimpleCNN, train_model, evaluate_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### 3.1 · Extract Droplet Patches

Uses the classical detector from Step 2 to locate droplets, then saves 64×64 crops.

In [ ]:
counts = build_patch_dataset()
for cls in CLASS_ORDER:
    print(f"  {cls:8s}: {counts[cls]} patches")
print(f"  Total   : {sum(counts.values())} patches  →  {PATCHES_DIR}/")

### 3.2 · Dataset Splits and DataLoaders

In [ ]:
train_ds, val_ds, test_ds = make_splits(val_frac=0.15, test_frac=0.15, seed=42)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

# show a batch of example patches
imgs_batch, labels_batch = next(iter(train_loader))
print(f"Batch shape: {imgs_batch.shape}  dtype: {imgs_batch.dtype}")

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, img, lbl in zip(axes.flat, imgs_batch[:16], labels_batch[:16]):
    patch = img.squeeze().numpy()
    patch = (patch * 0.5 + 0.5)   # undo normalisation for display
    ax.imshow(patch, cmap='gray', vmin=0, vmax=1)
    ax.set_title(IDX_TO_CLASS[lbl.item()], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample training patches (16 of first batch)', fontsize=11)
plt.tight_layout()
plt.show()

### 3.3 · Train SimpleCNN

20 epochs, Adam optimizer, CrossEntropyLoss. Prints progress every 5 epochs.

In [ ]:
model   = SimpleCNN(n_classes=len(CLASS_ORDER))
history = train_model(model, train_loader, val_loader, n_epochs=20, lr=1e-3, device=device)

test_loss, test_acc = evaluate_model(model, test_loader, device)
print(f"\nTest accuracy: {test_acc:.3f}  |  test loss: {test_loss:.4f}")

# save model weights
model_path = RESULTS_DIR / 'simple_cnn.pth'
torch.save(model.state_dict(), str(model_path))
print(f"Model saved  -> {model_path}")

### 3.4 · Training Curve

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, history['train_loss'], label='train', color='#4C9BE8')
axes[0].plot(epochs, history['val_loss'],   label='val',   color='#E8A44C')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-Entropy Loss')
axes[0].legend()

axes[1].plot(epochs, history['val_acc'], color='#5EBD70')
axes[1].axhline(test_acc, color='gray', linestyle='--', label=f'test acc = {test_acc:.3f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

plt.suptitle('Step 3 — SimpleCNN Training Curve', fontsize=12)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'training_curve.png'), dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved -> results/training_curve.png")

### 3.5 · Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# collect all test predictions
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds = model(imgs.to(device)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046)

ax.set_xticks(range(len(CLASS_ORDER)))
ax.set_yticks(range(len(CLASS_ORDER)))
ax.set_xticklabels(CLASS_ORDER)
ax.set_yticklabels(CLASS_ORDER)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Step 3 — Confusion Matrix (test set)')

# annotate each cell
for i in range(len(CLASS_ORDER)):
    for j in range(len(CLASS_ORDER)):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, cm[i, j], ha='center', va='center',
                fontsize=13, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'confusion_matrix.png'), dpi=130, bbox_inches='tight')
plt.show()

print("\nClassification report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_ORDER))
print("Saved -> results/confusion_matrix.png")

### 3.6 · Prediction Examples

5 test patches per class. Green border = correct, red border = wrong.

In [ ]:
from torchvision import transforms as T
from PIL import Image as PILImage

infer_transform = T.Compose([T.ToTensor(), T.Normalize(mean=[0.5], std=[0.5])])

def predict_patch(path):
    img    = PILImage.open(path).convert('L')
    tensor = infer_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        return model(tensor).argmax(1).item()

fig, axes = plt.subplots(3, 5, figsize=(12, 8))

for row_i, cls in enumerate(CLASS_ORDER):
    cls_pairs = [(p, l) for p, l in test_ds.pairs if l == CLASS_TO_IDX[cls]][:5]
    for col_i, (path, true_lbl) in enumerate(cls_pairs):
        pred_lbl = predict_patch(path)
        correct  = pred_lbl == true_lbl

        img = PILImage.open(path).convert('L')
        ax  = axes[row_i, col_i]
        ax.imshow(img, cmap='gray')
        ax.set_title(
            f"true: {IDX_TO_CLASS[true_lbl]}\npred: {IDX_TO_CLASS[pred_lbl]}",
            fontsize=8,
            color='green' if correct else 'red'
        )
        for spine in ax.spines.values():
            spine.set_edgecolor('green' if correct else 'red')
            spine.set_linewidth(2.5)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Step 3 — Prediction Examples  (green = correct  ·  red = wrong)',
             fontsize=11)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'prediction_examples.png'), dpi=130, bbox_inches='tight')
plt.show()
print("Saved -> results/prediction_examples.png")
print("Step 3 complete.")